# 01 · Exploratory Data Analysis
Target: `is_fraud` (0 = legitimate, 1 = fraud). Data: `data/raw/transactions.csv` (synthetic demo data unless you replaced it with your own CSV).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt, plotly.express as px
from src.config import DEFAULT_RAW_DATA, TARGET
from src.data_generation import generate_transactions
from src.preprocessing import load_dataset, clean_dataset, class_distribution, detect_outliers_iqr

if not DEFAULT_RAW_DATA.exists():
    DEFAULT_RAW_DATA.parent.mkdir(parents=True, exist_ok=True)
    generate_transactions().to_csv(DEFAULT_RAW_DATA, index=False)
df = load_dataset(DEFAULT_RAW_DATA)
df.head()

## 1 · Shape, types, missing values, duplicates

In [ ]:
print(df.shape)
display(df.dtypes.to_frame("dtype").join(df.isna().sum().rename("missing")))
print("exact duplicate rows:", df.duplicated().sum())

In [ ]:
df, report = clean_dataset(df)   # drops duplicates; outliers are NOT removed (they are often the fraud signal)
report

## 2 · Class distribution
Severe imbalance is why accuracy is useless here and why we use PR-AUC / recall / F1.

In [ ]:
dist = class_distribution(df); print(dist)
px.pie(df, names=df[TARGET].map({0: "Legitimate", 1: "Fraud"}), title="Class distribution", hole=.5)

## 3 · Outliers (reporting only)

In [ ]:
num_cols = ["amount", "distance_from_home_km", "txn_count_24h", "time_since_last_txn_min", "account_age_days"]
display(detect_outliers_iqr(df, num_cols).round(2))
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, c in zip(axes, ["amount", "distance_from_home_km", "time_since_last_txn_min"]):
    sns.boxplot(data=df, x=TARGET, y=c, ax=ax, showfliers=False); ax.set_yscale("log"); ax.set_title(c)
plt.tight_layout()

## 4 · Fraud vs legitimate behaviour

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"]); df["hour"] = df["timestamp"].dt.hour
hourly = df.groupby("hour")[TARGET].agg(["mean", "size"]).reset_index()
px.bar(hourly, x="hour", y="mean", title="Fraud rate by hour of day (night-time spike)")

In [ ]:
for col in ["merchant_category", "payment_channel", "device_type", "is_new_device", "country_mismatch"]:
    rate = df.groupby(col)[TARGET].mean().sort_values(ascending=False).reset_index()
    display(px.bar(rate, x=col, y=TARGET, title=f"Fraud rate by {col}"))

In [ ]:
px.histogram(df, x="amount", color=df[TARGET].map({0: "Legitimate", 1: "Fraud"}), log_y=True, nbins=60, barmode="overlay",
             title="Amount distribution by class")

In [ ]:
corr = df.select_dtypes("number").corr()
plt.figure(figsize=(8, 6)); sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size": 7}); plt.title("Correlation matrix")

**Takeaways** — write your own after running: which signals separate the classes, how heavy the imbalance is, and which columns need imputation.